In [1]:
import kagglehub
path = kagglehub.dataset_download(
    handle="saurabhbadole/game-of-thrones-book-dataset",
    force_download=False,
    output_dir="./data/GOT"
)
print("Path to dataset files:", path)

# Loading the dataset
with open("./data/GOT/1 - A Game of Thrones.txt") as f:
    data = f.read()

Path to dataset files: ./data/GOT


In [2]:
# Preprocessing
valid_chars = set("abcdefghijklmnopqrstuvwxyz 1234567890\n")
cleaned_data = "".join(char for char in data.lower() if char in valid_chars)
print(f"Removed characters: {''.join(sorted(set(data.lower()) - valid_chars))}")

Removed characters: !"'()*,-.:;?]`~


In [3]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

tokenized_data = word_tokenize(cleaned_data.lower())

[nltk_data] Downloading package punkt to /home/shriram/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/shriram/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
from collections import Counter

vocab = {'<PAD>': 0, '<UNK>': 1} # NOTE: Do not set negative token key → To find the empeddings, it only requires +ve integers
for index, word in enumerate(list(Counter(tokenized_data).keys())):
    vocab[word] = index + 2

print(len(vocab))

12513


In [5]:
# Splitting the data into sentences / documents
from typing import List
input_sentences = cleaned_data.split('\n')

def text_to_indices(sentences: str, vocab: dict) -> List[int]:
    words = sentences.strip().split(" ")
    tokens = []

    for word in words:
        token = vocab.get(word, vocab["<UNK>"])
        tokens.append(token)
    
    return tokens

encoded_data = []
for sentence in input_sentences:
    encoded_sentence = text_to_indices(sentence, vocab)
    if len(encoded_sentence) > 1: # NOTE: In Some sentences, there is only one token, which means there is nothing left for prediction.
        encoded_data.append(encoded_sentence)

len(encoded_data)

18971

In [6]:
# Prepering training dataset
def prepare_sentence(encoded_sentence: list) -> List[List[int]]:
    dataset = []
    for index in range(1, len(encoded_sentence)):
        dataset.append(encoded_sentence[:index+1])
    return dataset

prepared_dataset = []
max_len = 0
for sentence in encoded_data:
    prepared_sentence = prepare_sentence(sentence)
    prepared_dataset.extend(prepared_sentence)
    max_len = max(max_len, len(prepared_sentence[-1]))
    
len(prepared_dataset), max_len

(275841, 33)

In [7]:
import copy
pre_padded_dataset = copy.deepcopy(prepared_dataset) # Deep Copy because lists of lists.

# Performing global pre - padding on prepared dataset
for index, document in enumerate(prepared_dataset):
    padding_len = max_len - len(document)
    pre_padded_dataset[index] = ([0] * padding_len) + pre_padded_dataset[index]

print(pre_padded_dataset[0])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3]


In [8]:
# Splitting the dataset
import torch
from sklearn.model_selection import train_test_split

pre_padded_dataset = torch.tensor(pre_padded_dataset)
X = pre_padded_dataset[:, :-1].to(torch.int32)
y = pre_padded_dataset[:, -1:].to(torch.long) # labels must be in long datatype

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [9]:
from torch.utils.data import DataLoader, Dataset

class CustomDataset(Dataset):
    def __init__(self, X, y) -> None:
        super().__init__()
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, index):
        return self.X[index, :], self.y[index]

# Dataset Objects
train_dataset = CustomDataset(X_train, y_train)
test_dataset = CustomDataset(X_test, y_test)

# Data Loaders
train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=32, shuffle=True)

print(len(train_dataset), len(test_dataset))

220672 55169


In [10]:
# Model Architecture
from torch.nn import Module, RNN, Linear, Embedding

class Model(Module):
    def __init__(self, vocab_size:int) -> None:
        super().__init__()

        self.embedding = Embedding(num_embeddings=vocab_size, embedding_dim=64)
        self.rnn = RNN(input_size=64, hidden_size=50, batch_first=True)
        self.fc = Linear(in_features=50, out_features=vocab_size)

    def forward(self, input):
        embeddings = self.embedding(input) # converts the encodings to embeddings from 2D to 3D
        _, final_state = self.rnn(embeddings)
        output = self.fc(final_state.squeeze(0)) # squeezing num_hidden_layer dimension
        return output

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(len(vocab)).to(device)
model.train()

Model(
  (embedding): Embedding(12513, 64)
  (rnn): RNN(64, 50, batch_first=True)
  (fc): Linear(in_features=50, out_features=12513, bias=True)
)

In [12]:
from torch.nn import CrossEntropyLoss
from torch.optim import Adam

# Training Hyperparameters
epochs = 10
learning_rate = 0.001

# Loss and Optimizer
criterion = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=learning_rate)

In [13]:
for epoch in range(epochs):
    total_loss = 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()

        output = model(batch_x)
        loss = criterion(output, batch_y.flatten())
        loss.backward()

        optimizer.step()
        total_loss += loss.item() / len(batch_y)
    
    print(f"Epoch: {epoch + 1}, Loss: {total_loss / len(train_loader):.4f}")

Epoch: 1, Loss: 0.1956
Epoch: 2, Loss: 0.1761
Epoch: 3, Loss: 0.1688
Epoch: 4, Loss: 0.1640
Epoch: 5, Loss: 0.1603
Epoch: 6, Loss: 0.1572
Epoch: 7, Loss: 0.1546
Epoch: 8, Loss: 0.1522
Epoch: 9, Loss: 0.1501
Epoch: 10, Loss: 0.1482


In [38]:
def prediction(model: Model, vocab: dict, text):
    text = "".join(char for char in text.lower() if char in valid_chars)
    tokenized_text = word_tokenize(text)

    encoded_text = []
    for sentence in tokenized_text:
        encoded_sentence = text_to_indices(sentence, vocab)
        encoded_text.extend(encoded_sentence) # because text_to_indices returns list[int] and not int

    if len(encoded_text) <= max_len:
        padding_len = max_len - len(encoded_text)
        padded_text = torch.tensor(([0] * padding_len) + encoded_text).to(device)
        
        output = model(padded_text) # 1D

        _, index = torch.max(output, dim=0)

        # merge with text
        return text + " " + list(vocab.keys())[index]

In [41]:
num_tokens = 14
input_text = str("Finally Gared looked down. 'No fire,'")

for i in range(num_tokens):
    output_text = prediction(model, vocab, input_text)
    input_text = output_text

print(input_text)

finally gared looked down no fire and the other men had been a long time to be a good man


In [42]:
# Evaluation
def calculate_accuracy(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            outputs = model(batch_x)
            _, predicted = torch.max(outputs, dim=1)

            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

accuracy = calculate_accuracy(model)
print(f"Model Accuracy: {accuracy:.2f}%")

Model Accuracy: 78.43%
